# Fine-tune your 30M Wikipedia GPT into a Q&A model (v3)

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

v2 added more data variety (Dolly categories + Alpaca + real Wikipedia-grounded Q&A) and
early stopping, but training still collapsed into repeating a single token forever, at every
learning rate tried. The root cause turned out to be a **target-alignment bug**, not a data or
capacity problem: targets were built at the same position as the inputs instead of shifted by
one, so the model was being trained to predict *the token it was currently looking at* rather
than the next one. Because the embedding and output head are tied
(`head.weight = tok_emb.weight`), that's an almost-free shortcut to near-zero loss, which is
why training looked deceptively good (loss crashing fast, val loss near zero) while generation
produced nothing but blank lines - there's no "current token" to copy at inference time.

This version fixes that (Section 6) and keeps only what's load-bearing: broader/more varied
fine-tuning data (Section 4-5), correct target shifting (Section 6), and generation-checked
early stopping (Section 7) so a future bug like this would be caught immediately instead of
hiding behind a good-looking loss curve.

**What to upload first** (Colab file browser, left sidebar, or mount Drive):
- `latest.pt`  (your trained checkpoint)
- `tokenizer.json`  (the exact tokenizer used for pretraining)

Output: `finetuned.pt`, in the same checkpoint format your `generate.py` already loads.

In [ ]:
import torch, os
print(torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
assert torch.cuda.is_available(), "Go to Runtime -> Change runtime type -> T4 GPU, then re-run.

## 1. Upload your files
Run this cell, then use the file picker to upload `latest.pt` and `tokenizer.json`.
(If you'd rather mount Drive, comment this out and set `CKPT_PATH` / `TOKENIZER_PATH` below instead.)

In [ ]:
CKPT_PATH = "latest.pt"
TOKENIZER_PATH = "tokenizer.json"
assert os.path.exists(CKPT_PATH), "latest.pt not found - upload it (or fix CKPT_PATH)"
assert os.path.exists(TOKENIZER_PATH), "tokenizer.json not found - upload it (or fix TOKENIZER_PATH)

## 2. Model definition (must match `train_slm.ipynb` / `generate.py` exactly)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.n_embd, self.dropout = n_head, n_embd, dropout
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd, bias=False), nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50, repetition_penalty=1.0, stop_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            if repetition_penalty != 1.0:
                for token_id in set(idx_cond[0].tolist()):
                    if logits[0, token_id] > 0:
                        logits[0, token_id] /= repetition_penalty
                    else:
                        logits[0, token_id] *= repetition_penalty
            logits = logits / max(temperature, 1e-6)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            if stop_id is not None and next_id.item() == stop_id:
                break
        return idx

## 3. Load your pretrained checkpoint + tokenizer

In [ ]:
from tokenizers import Tokenizer

device = "cuda"
ck = torch.load(CKPT_PATH, map_location=device, weights_only=False)
cfg = ck["config"]
print("pretrained config:", cfg, "| pretrained step:", ck["step"])

model = GPT(cfg["vocab_size"], cfg["block_size"], cfg["n_layer"], cfg["n_head"], cfg["n_embd"], 0.0)
model.load_state_dict(ck["model"])
model.to(device)

tok = Tokenizer.from_file(TOKENIZER_PATH)
EOT = tok.token_to_id("<|endoftext|>")
assert EOT is not None, "tokenizer has no <|endoftext|> token - check you uploaded the right tokenizer.json"
BLOCK_SIZE = cfg["block_size"]
print("EOT id:", EOT, "| block size:", BLOCK_SIZE, "| vocab size:", cfg["vocab_size"])

## 4. Download + filter Dolly-15k + Alpaca

- **More Dolly categories kept** (`open_qa`, `closed_qa`, `general_qa`, `classification`,
  `summarization`, `brainstorming`), still capped on answer length, so the model sees more
  variety in question phrasing than just "what is X".
- **Alpaca-cleaned mixed in** (`yahma/alpaca-cleaned`, capped at `ALPACA_MAX`) for further
  phrasing diversity ("how did", "why does", "compare", "list") - GPT-3-generated so somewhat
  noisier than Dolly, hence the cap so it doesn't outweigh the more-reliable human-written data.

In [ ]:
!pip install -q -U datasets

from datasets import load_dataset

MAX_ANSWER_WORDS = 60     # keep answers short - this model can't do long reasoning
MAX_CONTEXT_WORDS = 80    # for examples that include a context passage
KEEP_DOLLY_CATEGORIES = {"open_qa", "closed_qa", "general_qa", "classification",
                          "summarization", "brainstorming"}
ALPACA_MAX = 4000          # cap so Alpaca doesn't outweigh the more-reliable Dolly data

def word_ok(text, cap):
    return len(text.split()) <= cap

# ---- Dolly ----
raw_dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
print("raw dolly examples:", len(raw_dolly))

examples = []
for ex in raw_dolly:
    if ex["category"] not in KEEP_DOLLY_CATEGORIES:
        continue
    if not ex["response"].strip() or not ex["instruction"].strip():
        continue
    if not word_ok(ex["response"], MAX_ANSWER_WORDS):
        continue
    ctx = ex["context"].strip()
    if ctx and not word_ok(ctx, MAX_CONTEXT_WORDS):
        continue
    examples.append({"instruction": ex["instruction"].strip(), "context": ctx,
                      "response": ex["response"].strip(), "source": "dolly"})

print("filtered dolly examples:", len(examples))

# ---- Alpaca (cleaned) ----
raw_alpaca = load_dataset("yahma/alpaca-cleaned", split="train")
print("raw alpaca-cleaned examples:", len(raw_alpaca))

alpaca_examples = []
for ex in raw_alpaca:
    instr = ex["instruction"].strip()
    resp = ex["output"].strip()
    ctx = ex.get("input", "").strip()
    if not instr or not resp:
        continue
    if not word_ok(resp, MAX_ANSWER_WORDS):
        continue
    if ctx and not word_ok(ctx, MAX_CONTEXT_WORDS):
        continue
    alpaca_examples.append({"instruction": instr, "context": ctx, "response": resp, "source": "alpaca"})

import random
random.Random(1337).shuffle(alpaca_examples)
alpaca_examples = alpaca_examples[:ALPACA_MAX]
print("filtered + capped alpaca examples:", len(alpaca_examples))

examples = examples + alpaca_examples
print("total examples so far:", len(examples))
for e in examples[:3]:
    print("-" * 40)
    print(e)

## 5. Wikipedia-grounded Q&A

Streams a few thousand short Wikipedia articles (same source dataset as pretraining,
`wikimedia/wikipedia` / `20231101.en`) and turns each into a `Who is {title}?` /
`What is {title}?` pair using the article's own opening sentence as the answer - so at least
some fine-tuning examples are guaranteed to be facts the base model actually saw in
pretraining. Runs on CPU via streaming; no Drive-mounted `.bin` files needed.

In [ ]:
N_WIKI_ARTICLES = 4000   # how many articles to scan (not all will pass the length filter)
MIN_PARA_WORDS = 8

wiki_ds = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
wiki_ds = wiki_ds.shuffle(seed=1337, buffer_size=2000)

wiki_qa = []
for i, ex in enumerate(wiki_ds):
    if i >= N_WIKI_ARTICLES:
        break
    title = ex["title"].strip()
    text = ex["text"].strip()
    if not title or not text:
        continue
    first_para = text.split("\n\n")[0].strip()
    n_words = len(first_para.split())
    if n_words < MIN_PARA_WORDS or n_words > MAX_ANSWER_WORDS:
        continue
    question = f"Who is {title}?" if first_para[:1].isupper() and not first_para.lower().startswith(("the ", "a ", "an ")) else f"What is {title}?"
    wiki_qa.append({"instruction": question, "context": "", "response": first_para, "source": "wiki"})

print(f"generated {len(wiki_qa)} Wikipedia-grounded Q&A pairs from {N_WIKI_ARTICLES} scanned articles")
for e in wiki_qa[:3]:
    print("-" * 40)
    print(e)

examples = examples + wiki_qa
print("\ntotal training examples:", len(examples))

from collections import Counter
print("by source:", Counter(e["source"] for e in examples))

## 6. Tokenize + build train/val split (correct next-token shift)

**This is the fix.** Targets must be the input sequence shifted one position to the left
(`targets[t] = inputs[t+1]`), because `GPT.forward` computes loss position-for-position with
no internal shift. The prompt is masked out of the loss except its very last position, whose
target is the first answer token - that's the one transition we actually want the model to
learn ("given the full question, predict the start of the answer"). Getting this wrong (as the
previous version did - `targets` aligned with `ids` instead of shifted) lets the model minimize
loss by predicting the token it's currently looking at, which the tied embedding/head matrix
makes almost free - it looks like fast, clean convergence right up until you try to generate
from it, at which point there's no "current token" to copy and it collapses.

In [ ]:
import random

def build_example(ex):
    if ex["context"]:
        prompt = f"### Context:\n{ex['context']}\n\n### Question:\n{ex['instruction']}\n\n### Answer:\n"
    else:
        prompt = f"### Question:\n{ex['instruction']}\n\n### Answer:\n"
    answer = ex["response"] + "<|endoftext|>"

    prompt_ids = tok.encode(prompt).ids
    answer_ids = tok.encode(answer).ids
    full = prompt_ids + answer_ids
    if len(full) > BLOCK_SIZE or len(prompt_ids) == 0:
        return None

    # standard next-token shift: inputs are everything but the last token, targets are
    # everything but the first. The prompt stays masked except its final position, whose
    # target is the first answer token.
    inputs = full[:-1]
    targets = [-100] * (len(prompt_ids) - 1) + answer_ids
    assert len(inputs) == len(targets)
    return inputs, targets

random.Random(1337).shuffle(examples)
tokenized = [t for t in (build_example(e) for e in examples) if t is not None]
print(f"tokenized examples: {len(tokenized)} (dropped {len(examples) - len(tokenized)} for exceeding block size {BLOCK_SIZE})")

n_val = max(1, int(0.05 * len(tokenized)))
val_data = tokenized[:n_val]
train_data = tokenized[n_val:]
print(f"train: {len(train_data)} | val: {len(val_data)}")

In [ ]:
PAD_ID = EOT  # padding re-uses <|endoftext|>; padded positions are masked out of the loss below

def collate(batch):
    maxlen = max(len(x[0]) for x in batch)
    x = torch.full((len(batch), maxlen), PAD_ID, dtype=torch.long)
    y = torch.full((len(batch), maxlen), -100, dtype=torch.long)
    for i, (ids, targets) in enumerate(batch):
        x[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        y[i, :len(targets)] = torch.tensor(targets, dtype=torch.long)
    return x.to(device), y.to(device)

def get_batches(data, batch_size, shuffle=True):
    idx = list(range(len(data)))
    if shuffle:
        random.shuffle(idx)
    for i in range(0, len(idx), batch_size):
        chunk = [data[j] for j in idx[i:i + batch_size]]
        yield collate(chunk)

## 7. Fine-tune, with generation-checked early stopping

Teacher-forced val loss alone can't detect a model that's collapsed into repeating one token -
loss is computed with the true previous token always available, so a broken model can still
look fine on that metric alone. Every eval, this also runs real free-running generation on a
few probe questions and only accepts a checkpoint as "best" if it's both lower-loss *and*
producing non-degenerate text (not blank/whitespace/a single repeated character).

In [ ]:
import copy, time

EPOCHS = 6
BATCH_SIZE = 16
LR = 5e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
EVAL_EVERY_STEPS = 50
PATIENCE_EVALS = 4

PROBE_QUESTIONS = ["Who is Joan of Arc?", "How did Stalin die?", "What is the capital of France?"]

def is_degenerate(text):
    stripped = text.strip()
    if len(stripped) < 3:
        return True
    if len(set(stripped.replace("\n", "").replace(" ", ""))) <= 1:
        return True
    return False

@torch.no_grad()
def generate_probe(question, max_new_tokens=40):
    prompt = f"### Question:\n{question}\n\n### Answer:\n"
    ids = tok.encode(prompt).ids
    ctx = torch.tensor([ids], dtype=torch.long, device=device)
    model.eval()
    out = model.generate(ctx, max_new_tokens, temperature=0.7, top_k=40, repetition_penalty=1.3, stop_id=EOT)[0].tolist()
    answer_ids = out[len(ids):]
    if answer_ids and answer_ids[-1] == EOT:
        answer_ids = answer_ids[:-1]
    model.train()
    return tok.decode(answer_ids)

@torch.no_grad()
def estimate_val_loss():
    model.eval()
    losses = []
    for xb, yb in get_batches(val_data, BATCH_SIZE, shuffle=False):
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return sum(losses) / max(1, len(losses))

decay, no_decay = [], []
for n, p in model.named_parameters():
    (no_decay if p.dim() < 2 else decay).append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}],
    lr=LR, betas=(0.9, 0.95),
)

best_val = estimate_val_loss()
best_state = copy.deepcopy(model.state_dict())
best_step = 0
evals_since_improve = 0
step = 0
stop = False
t0 = time.time()

print("val loss before fine-tuning:", f"{best_val:.4f}")
print("probes before fine-tuning:")
for q in PROBE_QUESTIONS:
    print(f"  Q: {q}\n  A: {generate_probe(q)!r}")

model.train()
for epoch in range(EPOCHS):
    if stop:
        break
    for xb, yb in get_batches(train_data, BATCH_SIZE, shuffle=True):
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(xb, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        step += 1

        if step % 20 == 0:
            print(f"epoch {epoch+1}/{EPOCHS} | step {step} | loss {loss.item():.4f} "
                  f"| {(time.time()-t0)/60:.1f} min elapsed")

        if step % EVAL_EVERY_STEPS == 0:
            vl = estimate_val_loss()
            probes = {q: generate_probe(q) for q in PROBE_QUESTIONS}
            n_degenerate = sum(is_degenerate(a) for a in probes.values())

            print(f"\n  [eval] step {step} | val loss {vl:.4f} | degenerate: {n_degenerate}/{len(PROBE_QUESTIONS)}")
            for q, a in probes.items():
                print(f"    Q: {q}\n    A: {a!r}")

            if vl < best_val and n_degenerate == 0:
                best_val, best_state, best_step, evals_since_improve = vl, copy.deepcopy(model.state_dict()), step, 0
                print(f"    -> new best (val loss {best_val:.4f})")
            else:
                evals_since_improve += 1
                reason = " [degenerate output]" if n_degenerate else ""
                print(f"    -> no improvement ({evals_since_improve}/{PATIENCE_EVALS}){reason}")
                if evals_since_improve >= PATIENCE_EVALS:
                    print(f"\nno improvement in {PATIENCE_EVALS} evals - stopping early at step {step}.")
                    stop = True
                    break
    else:
        continue
    break

print(f"\ndone in {(time.time()-t0)/60:.1f} min | best val loss {best_val:.4f} at step {best_step}")
if best_step == 0:
    print("WARNING: no checkpoint ever beat the pre-finetuning baseline without degenerating - "
          "something is still off (check Section 6's target alignment first).")
model.load_state_dict(best_state)
print("loaded best checkpoint (step", best_step, ") into model for testing/saving")

## 8. Try it before saving

In [ ]:
def ask(question, context="", max_new_tokens=100, temperature=0.7, top_k=40, rep_penalty=1.3):
    if context:
        prompt = f"### Context:\n{context}\n\n### Question:\n{question}\n\n### Answer:\n"
    else:
        prompt = f"### Question:\n{question}\n\n### Answer:\n"
    ids = tok.encode(prompt).ids
    ctx = torch.tensor([ids], dtype=torch.long, device=device)
    model.eval()
    out = model.generate(ctx, max_new_tokens, temperature, top_k, rep_penalty, stop_id=EOT)[0].tolist()
    answer_ids = out[len(ids):]
    if answer_ids and answer_ids[-1] == EOT:
        answer_ids = answer_ids[:-1]
    print("Q:", question)
    print("A:", tok.decode(answer_ids))
    print()

ask("Who is Joan of Arc?")
ask("How did Stalin die?")
ask("What is the capital of France?")
ask("Explain what photosynthesis is.")
ask("Summarize the French Revolution in one sentence.")

## 9. Save `finetuned.pt`
Saves the **best-val-loss, non-degenerate checkpoint** (loaded back into `model` in Section 7). Same format your `generate.py` / `train_slm.ipynb` already use.

In [ ]:
SAVE_PATH = "finetuned.pt"
torch.save({
    "step": ck["step"] + best_step,
    "best_val": best_val,
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "config": cfg,
}, SAVE_PATH)
print("saved", SAVE_PATH, f"({os.path.getsize(SAVE_PATH)/1e6:.1f} MB)", "| best val loss:", f"{best_val:.4f}")

from google.colab import files
files.download(SAVE_PATH)

## Next step

Download `finetuned.pt` locally, next to your `generate.py` and `tokenizer.json` - it's a
drop-in replacement, `generate.py` doesn't need any changes for this version.

If answers are grammatical but still factually wrong on things like "How did Stalin die?", that
gap is base-model capacity/knowledge (a 30M model pretrained on ~1B tokens has very little room
to memorize biographical facts), not a training bug - fine-tuning can only surface facts the
base model already picked up in pretraining. Increasing `N_WIKI_ARTICLES` in Section 5 gives it
more grounded facts to draw on; a bigger pretrained base model (more layers/params, more
pretraining tokens) would be the other lever, and would use this exact same fine-tuning
notebook unchanged.